In [ ]:
import sunpy
import numpy as np
from math import *
import astropy.units as u
from astropy.io import fits
import matplotlib.pyplot as plt
from sunpy.coordinates import frames
import matplotlib.gridspec as gridspec
from scipy.interpolate import interp1d,RegularGridInterpolator
from astropy.coordinates import SkyCoord
from scipy.io import readsav
from matplotlib.patches import Rectangle
from scipy.ndimage import zoom
from astropy.wcs import WCS
from astropy.wcs.utils import pixel_to_skycoord
import cv2
import glob
import os
from astropy.coordinates import SkyCoord
from sunpy.physics.differential_rotation import differential_rotate
from sunpy.coordinates import propagate_with_solar_surface
import pandas as pd
import shutil

In [ ]:
#这段代码是把一块下好的AIA不同波段分出来
list1=glob.glob(r'C:\Learning\PHD2nd\sunspotscar\data\to_be_exam\20140213_013200_UTC\JSOC_20260827_000421\*.fits')
print(li)

In [ ]:
#裁剪目标区域并进行旋转对齐
root_dir=r'C:\Learning\PHD2nd\sunspotscar\data\M'
file_time=os.listdir(root_dir)
df=pd.read_excel('../dataset/M级耀斑.xlsx')
#df.drop(df.index[8:16], inplace=True)
x_range=df['X, arcsec'].values
y_range=df['Y, arcsec'].values
passbands=['131','211','304','hmi.Ic_45s']
width=800
for i in range(59,100):
    for j in passbands:
        file_name=glob.glob(os.path.join(root_dir,file_time[i],j,'*.fits'))
        os.makedirs(os.path.join(root_dir,file_time[i],j+'sub_map'), exist_ok=True)
        sub_file_path=os.path.join(root_dir,file_time[i],j+'sub_map')
        for k in range(len(file_name)):
            if k==0:
                map0=sunpy.map.Map(file_name[k])
                bl1=SkyCoord((x_range[i]-width//2)*u.arcsec,(y_range[i]-width//2)*u.arcsec, frame=map0.coordinate_frame)
                tr1=SkyCoord((x_range[i]+width//2)*u.arcsec,(y_range[i]+width//2)*u.arcsec, frame=map0.coordinate_frame)
                sub_map0=map0.submap(bl1,top_right=tr1)
                sub_map0.meta.pop('BLANK', None)
                sub_map0.save(os.path.join(sub_file_path,file_name[k][-38:]),overwrite=True)
            else:
                map1=sunpy.map.Map(file_name[k])
                with propagate_with_solar_surface():
                    rot_map1=map1.reproject_to(sub_map0.wcs,preserve_date_obs=True)
                for key in ['telescop', 'instrume', 'detector', 'wavelnth', 'waveunit',
                            'bunit', 'exptime']:
                    if key in map1.meta:
                        rot_map1.meta[key] = map1.meta[key]

                rot_map1 = sunpy.map.Map(rot_map1.data.astype('float32'), rot_map1.meta)
                rot_map1.meta.pop('BLANK', None)
                rot_map1.save(os.path.join(sub_file_path, file_name[k][-38:]), overwrite=True)

#裁剪Br
for i in range(59,100):
    sub_file_path=os.path.join(root_dir,file_time[i],'hmi.B_720s')
    try:
        map0=sunpy.map.Map(sub_file_path+'\\'+'Br.fits')
        bl1=SkyCoord((x_range[i]-width//2)*u.arcsec,(y_range[i]-width//2)*u.arcsec, frame=map0.coordinate_frame)
        tr1=SkyCoord((x_range[i]+width//2)*u.arcsec,(y_range[i]+width//2)*u.arcsec, frame=map0.coordinate_frame)
        sub_map=map0.submap(bl1,top_right=tr1)
        sub_map.meta.pop('BLANK', None)
        sub_map.save(os.path.join(root_dir,file_time[i],'hmi.B_720s','Br_sub.fits'),overwrite=True)
    except:
        print(file_time[i]+'hmi.B_720s'+'Br.fits'+'不存在')